In [12]:
import time
import json
import os
import subprocess
from datetime import datetime
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Dynamische IP & Modell-Setup nach Best Practices
windows_ip = subprocess.check_output("ip route list default | awk '{print $3}'", shell=True).decode('utf-8').strip()

# Exakte Parameter für gemma4:e4b
model = ChatOllama(
    model="gemma4:latest", 
    base_url=f"http://{windows_ip}:11434", 
    temperature=1.0, 
    top_p=0.95, 
    top_k=64
)

print(f"✅ Backend verbunden. Modell: gemma4:e4b | Temp: 1.0")

✅ Backend verbunden. Modell: gemma4:e4b | Temp: 1.0


In [13]:
# 2. Der native Tracking-Motor (Korrigiert für langchain-ollama)
def run_native_reasoning_prompt(system_instruction, user_prompt, step_name, max_tokens):
    print(f"--- Führe aus: {step_name} (Hard Limit: {max_tokens} Output Tokens) ---")
    start_time = time.time()
    
    # FIX: Wir deklarieren das Modell direkt hier mit dem spezifischen num_predict für diesen Lauf
    local_model = ChatOllama(
        model="gemma4:latest", 
        base_url=f"http://{windows_ip}:11434", 
        temperature=1.0, 
        top_p=0.95, 
        top_k=64,
        num_predict=max_tokens  # Die Notbremse direkt verankert
    )
    
    messages = [
        SystemMessage(content=system_instruction),
        HumanMessage(content=user_prompt)
    ]
    
    # Aufruf ohne .bind(), da die Parameter schon im local_model stecken
    response = local_model.invoke(messages)
    raw_output = response.content
    
    total_time = round(time.time() - start_time, 2)
    meta = response.response_metadata
    
    thought_process = ""
    final_answer = raw_output
    
    if "<|channel>thought" in raw_output and "<channel|>" in raw_output:
        parts = raw_output.split("<channel|>")
        thought_process = parts[0].replace("<|channel>thought\n", "").replace("<|channel>thought", "").strip()
        final_answer = parts[1].strip() if len(parts) > 1 else ""
    
    print(f"Fertig in {total_time}s | Tokens: {meta.get('prompt_eval_count', 0)} In / {meta.get('eval_count', 0)} Out")
    
    return {
        "step_name": step_name,
        "system_prompt": system_instruction,
        "latency_seconds": total_time,
        "input_tokens": meta.get('prompt_eval_count', 0),
        "output_tokens": meta.get('eval_count', 0),
        "thought_process": thought_process,
        "final_answer": final_answer,
        "raw_output": raw_output
    }

In [14]:
# ==========================================
# DAS EXPERIMENT: NATIVE REASONING VS BASELINE
# ==========================================

context_text = "Exactly six trade representatives negotiate a treaty: Klosnik, Londi, Manley, Neri, Osata, Poirier. There are exactly six chairs evenly spaced around a circular table. The chairs are numbered 1 through 6, with successively numbered chairs next to each other and chair number 1 next to chair number 6. Each chair is occupied by exactly one of the representatives. The following conditions apply: Poirier sits immediately next to Neri. Londi sits immediately next to Manley, Neri, or both. Klosnik does not sit immediately next to Manley. If Osata sits immediately next to Poirier, Osata does not sit immediately next to Manley."
question_text = "Generate exactly one valid seating arrangement of the six representatives in chairs 1 through 6 that does NOT violate the stated conditions."

experiment_data = {
    "timestamp": datetime.now().isoformat(),
    "model": "gemma4:e4b",
    "context": context_text,
    "question": question_text,
    "trials": []
}

user_task = f"""Context: {context_text}
Question: {question_text}
Task: ONLY output the final sequence as a comma-separated list. Example format: Name1, Name2, Name3, Name4, Name5, Name6"""

# ---------------------------------------------------------
# Versuch 1: Baseline (Disabled Thinking)
# ---------------------------------------------------------
# Kein <|think|> Tag im System-Prompt. 
baseline_system = "You are a logical solver. Adhere strictly to output formats."
result_baseline = run_native_reasoning_prompt(baseline_system, user_task, "Baseline (No Thinking)", max_tokens=20)
experiment_data["trials"].append(result_baseline)

# ---------------------------------------------------------
# Versuch 2: Native Reasoning (Trigger Thinking)
# ---------------------------------------------------------
# Der magische Trigger ist hier ganz vorne platziert
reasoning_system = "<|think|> You are a logical solver. Think step-by-step internally before answering."
# Wir geben der GPU hier 500 Tokens Spielraum, da Denken und Antwort in einem Stream kommen
result_reasoning = run_native_reasoning_prompt(reasoning_system, user_task, "Native Reasoning", max_tokens=500)
experiment_data["trials"].append(result_reasoning)


# JSON Log speichern
log_file = "gemma_experiment_log.json"
if os.path.exists(log_file):
    with open(log_file, "r", encoding="utf-8") as f:
        log_data = json.load(f)
else:
    log_data = []

log_data.append(experiment_data)

with open(log_file, "w", encoding="utf-8") as f:
    json.dump(log_data, f, indent=4, ensure_ascii=False)

print("\n--- ERGEBNISSE ---")
print(f"Baseline Antwort: {result_baseline['final_answer']}")
print(f"Reasoning Antwort: {result_reasoning['final_answer']}")

--- Führe aus: Baseline (No Thinking) (Hard Limit: 20 Output Tokens) ---
Fertig in 145.0s | Tokens: 236 In / 20 Out
--- Führe aus: Native Reasoning (Hard Limit: 500 Output Tokens) ---


KeyboardInterrupt: 